In [1]:
!pip install yfinance pandas numpy scikit-learn joblib pmdarima prophet tensorflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 6.6 MB/s eta 0:00:00


In [2]:
import joblib
import numpy as np
import pandas as pd
import yfinance as yf

from pmdarima import auto_arima
from prophet import Prophet
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

tickers = ["ETH-USD", "USDT-USD"]

for ticker in tickers:

    print(f"\n========== TRAINING {ticker} ==========")

    slug = ticker.split("-")[0].lower()

    # =========================
    # Download Data
    # =========================

    df = yf.download(
        ticker,
        period="max",
        interval="1d",
        progress=False
    )

    # Fix yfinance multiindex issue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.dropna(inplace=True)

    # =========================
    # Feature Engineering
    # =========================

    df["log_return"] = np.log(
        df["Close"] / df["Close"].shift(1)
    )

    df["ma7"] = df["Close"].rolling(7).mean()

    df["ma21"] = df["Close"].rolling(21).mean()

    df["volatility_7"] = (
        df["log_return"].rolling(7).std()
    )

    # RSI

    delta = df["Close"].diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(14).mean()

    avg_loss = loss.rolling(14).mean()

    rs = avg_gain / avg_loss

    df["rsi14"] = 100 - (100 / (1 + rs))

    df.dropna(inplace=True)

    # =========================
    # Train ARIMA
    # =========================

    print("Training ARIMA...")

    arima_model = auto_arima(
        df["Close"],
        seasonal=False,
        stepwise=True,
        suppress_warnings=True
    )

    joblib.dump(
        arima_model,
        f"{slug}_arima.pkl"
    )

    print("ARIMA done")

    # =========================
    # Train Prophet
    # =========================

    print("Training Prophet...")

    prophet_df = pd.DataFrame({
        "ds": df.index,
        "y": df["Close"].squeeze().values
    })

    prophet_model = Prophet()

    prophet_model.fit(prophet_df)

    joblib.dump(
        prophet_model,
        f"{slug}_prophet.pkl"
    )

    print("Prophet done")

    # =========================
    # Prepare LSTM Data
    # =========================

    FEATURE_COLUMNS = [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "log_return",
        "ma7",
        "ma21",
        "volatility_7",
        "rsi14"
    ]

    scaler = StandardScaler()

    X_raw = df[FEATURE_COLUMNS].values

    y = df["log_return"].values

    X_scaled = scaler.fit_transform(X_raw)

    window_size = 60

    X = []
    y_seq = []

    for i in range(window_size, len(X_scaled)):

        X.append(
            X_scaled[i-window_size:i]
        )

        y_seq.append(y[i])

    X = np.array(X)

    y_seq = np.array(y_seq)

    # =========================
    # Train/Test Split
    # =========================

    split = int(len(X) * 0.8)

    X_train = X[:split]
    X_test = X[split:]

    y_train = y_seq[:split]
    y_test = y_seq[split:]

    # =========================
    # Build LSTM
    # =========================

    model = Sequential()

    model.add(
        LSTM(
            64,
            return_sequences=True,
            input_shape=(
                X.shape[1],
                X.shape[2]
            )
        )
    )

    model.add(Dropout(0.2))

    model.add(LSTM(32))

    model.add(Dropout(0.2))

    model.add(Dense(1))

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    # =========================
    # Train LSTM
    # =========================

    print("Training LSTM...")

    model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=32
    )

    model.save(
        f"{slug}_lstm.h5"
    )

    joblib.dump(
        scaler,
        f"{slug}_scaler.pkl"
    )

    print(f"{ticker} training completed")


========== TRAINING ETH-USD ==========


/tmp/ipykernel_18798/2779260631.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(


Training ARIMA...
ARIMA done
Training Prophet...


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Prophet done
Training LSTM...
Epoch 1/20


...

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


ARIMA done
Training Prophet...
Prophet done
Training LSTM...
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


76/76 ━━━━━━━━━━━━━━━━━━━━ 7s 57ms/step - loss: 0.0058 - val_loss: 1.7589e-04
Epoch 2/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - loss: 0.0016 - val_loss: 1.0574e-04
Epoch 3/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - loss: 9.0525e-04 - val_loss: 6.0427e-05
Epoch 4/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - loss: 5.4889e-04 - val_loss: 1.4974e-04
Epoch 5/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 5s 65ms/step - loss: 4.6413e-04 - val_loss: 2.4544e-05
Epoch 6/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 3.0709e-04 - val_loss: 1.4145e-05
Epoch 7/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - loss: 2.8721e-04 - val_loss: 4.5668e-06
Epoch 8/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step - loss: 2.1904e-04 - val_loss: 5.1321e-06
Epoch 9/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - loss: 1.8919e-04 - val_loss: 8.8486e-06
Epoch 10/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - loss: 1.6311e-04 - val_loss: 5.9141e-06
Epoch 11/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 1.4529e-04 - val_loss: 7.086

USDT-USD training completed
